# gramlab · run 22 — encode · calculate · decode with Holographic Reduced Representations

The architecture question left open since the Nov‑22 pilot: can a record be *calculated over* and still give back the
specific tokens? Plate's HRR on the project's own records (AMALGUM, gold roles; live roles from the learned linker for a
precision check). Codebook = deterministic random unit vectors per symbol (layer 0 needs no storage); record =
ACTOR⊛actor + ACTION⊛action + PATIENT⊛patient + STATUS⊛status; ⊛ = circular convolution.

Measured: (1) **capacity** — superpose k records, unbind each role, is the filler in the top‑k cleanup, for d = 512…4096;
(2) **reversal** — record vs role‑swapped twin: cosine, and whether ACTOR decodes to the twin's actor (by construction it
should); (3) **questions** — "who V‑ed X?" over 20k stored record vectors by probe similarity + unbinding, against the
exact key lookup on the same questions; (4) **cost** — bytes/record and µs/question. CPU only, ~20–30 min.

In [ ]:
import os, sys, glob, subprocess, pathlib, json
WORK = pathlib.Path('/kaggle/working'); os.chdir(WORK)
def sh(c):
    print('$', c); r = subprocess.run(c, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-1500:], r.stderr[-1500:])
    return r.returncode == 0
sh('git clone --depth 1 https://github.com/furevych/gramlab.git gramlab')
sys.path.insert(0, str(WORK / 'gramlab'))
GUM = WORK / 'data' / 'gum'; GUM.mkdir(parents=True, exist_ok=True)
for split in ('train', 'dev', 'test'):
    sh(f'curl -sSL -o {GUM}/en_gum-ud-{split}.conllu https://raw.githubusercontent.com/UniversalDependencies/UD_English-GUM/master/en_gum-ud-{split}.conllu')
AMALGUM = 'data/amalgum/amalgum/*/dep/*.conllu'
if not glob.glob(AMALGUM):
    sh('GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/gucorpling/amalgum.git data/amalgum')
print('GUM', sorted(p.name for p in GUM.glob('*.conllu')), '| AMALGUM files', len(glob.glob(AMALGUM)))
# layer-0 preset: WordNet 3.0 files, read by gramlab.core.wordnet (no nltk needed)
if not pathlib.Path('data/wordnet/index.verb').exists():
    sh('curl -sSL -o data/wordnet.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/wordnet.zip')
    import zipfile; zipfile.ZipFile('data/wordnet.zip').extractall('data')      # -> data/wordnet/
os.environ['WORDNET_DIR'] = str(WORK / 'data' / 'wordnet')
from gramlab.core.wordnet import WordNet; print('WordNet', WordNet().stats())
if not pathlib.Path('data/semcor/brown2').exists():
    sh('curl -sSL -o data/semcor.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/semcor.zip')
    zipfile.ZipFile('data/semcor.zip').extractall('data')                        # -> data/semcor/
SEMCOR = str(WORK / 'data' / 'semcor')
# affect lexicon: attach a Kaggle dataset containing NRC-VAD-Lexicon.txt (word, valence, arousal, dominance) to enable the affect row
VAD = next(iter(glob.glob('/kaggle/input/**/NRC-VAD-Lexicon*.txt', recursive=True) + glob.glob('/kaggle/input/**/*vad*.txt', recursive=True)), None)
print('VAD lexicon:', VAD)


In [ ]:
from gramlab.core import hrr
import json
board = hrr.run(AMALGUM, out_dir=str(WORK / 'results'), dims=(512, 1024, 2048, 4096), ks=(1, 4, 16, 64, 256), limit=60000)
print(json.dumps({'reversal': board['reversal'], 'questions': board['questions']}, indent=1))

## Reading it
- `capacity`: how many records a single superposed memory can hold at each d before the specific fillers stop coming back.
  This is the cost of *calculating over a sum*; it is a known limit of HRR, and the number to compare against the key's zero‑loss postings.
- `reversal`: must be ~1.0 — order is in the binding, not learned.
- `questions`: HRR top‑1 actor vs key first posting on identical questions; ties on genuinely ambiguous (verb, patient) pairs are expected.
Record as run 22 in RESULTS.md.